# 05 — Forecasting Models

One **global** LightGBM across all series, not one model per SKU: most series are too sparse to support their own model, new products have no history at all, and per-series fitting costs scale with the number of series for no additional insight.

Objective is Tweedie because demand is a non-negative, zero-inflated count; squared error treats it as symmetric and unbounded.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

panel: 1,003,600 rows, 600 series, 1969 days


## Walk-forward validation with refitting per fold

Runs several minutes: the model is refitted for each fold, because reusing one model fitted on all training data would let fold 1 be scored by a model that had seen fold 4.

In [2]:
from src.evaluation.validate import walk_forward, aggregate_scores, per_fold_table
from src.evaluation.metrics import skill_vs
res = walk_forward(cfg, panel, fit_ml=True, n_origins=20)
skill_vs(aggregate_scores(res), 'seasonal_naive').round(4)

  fold_1   origin 2016-01-31 predict 2016-02-01..2016-02-28 | train rows 336,000 | best moving_average_28 WAPE 0.7635 | 78s


  fold_2   origin 2016-02-28 predict 2016-02-29..2016-03-27 | train rows 336,000 | best lightgbm WAPE 0.7549 | 76s


  fold_3   origin 2016-03-27 predict 2016-03-28..2016-04-24 | train rows 336,000 | best lightgbm WAPE 0.7170 | 78s


  fold_4   origin 2016-04-24 predict 2016-04-25..2016-05-22 | train rows 336,000 | best moving_average_28 WAPE 0.6977 | 77s


,model,folds,mae,rmse,wape,wape_std,mase,smape,bias,skill_vs_seasonal_naive_pct
0,lightgbm,4,1.0352,2.3058,0.7426,0.0448,0.9487,142.5545,-0.0534,14.3258
1,moving_average_28,4,1.0455,2.2941,0.7503,0.0402,0.9492,132.0348,-0.0403,13.4363
2,random_forest,4,1.0514,2.2707,0.7543,0.0348,0.9659,141.3685,-0.0362,12.9708
3,moving_average_7,4,1.0536,2.2890,0.7558,0.0323,0.9582,115.4773,-0.0653,12.8049
4,sba,4,1.1001,2.3672,0.7901,0.0606,0.9796,142.9735,-0.0110,8.8489
5,croston,4,1.1150,2.3796,0.8008,0.0617,0.9941,142.2168,0.0619,7.6130
6,seasonal_naive,4,1.2083,2.7940,0.8668,0.0396,1.0670,80.1448,-0.0653,0.0000
7,zero,4,1.3958,3.8260,1.0000,0.0000,0.9420,87.8750,-1.3958,-15.3725
8,naive,4,1.4090,3.1630,1.0105,0.0817,1.1974,83.0502,0.3421,-16.5782


In [3]:
per_fold_table(res).round(4)

,fold_1,fold_2,fold_3,fold_4
naive,1.1160,1.0273,0.9259,0.9726
seasonal_naive,0.9051,0.8948,0.8442,0.8229
moving_average_7,0.7837,0.7776,0.7487,0.7131
moving_average_28,0.7635,0.7938,0.7463,0.6977
croston,0.8179,0.8703,0.7931,0.7218
sba,0.8094,0.8561,0.7838,0.7110
zero,1.0000,1.0000,1.0000,1.0000
lightgbm,0.7998,0.7549,0.7170,0.6986
random_forest,0.7735,0.7865,0.7501,0.7072


## Feature importance

In [4]:
from src.features.build import build_supervised, training_origins
from src.forecasting import ml as ML
from src.data.loader import rolling_origin_folds
fold = rolling_origin_folds(cfg, panel)[-1]
tr = build_supervised(cfg, panel, fold,
                      origins=training_origins(cfg, panel, fold.origin, 20))
m = ML.fit_lightgbm(cfg, tr)
ML.feature_importance(m, top=15)

,feature,importance
0,rolling_mean_28,3.776732e+06
1,rolling_mean_14,1.348149e+06
2,sku_id,7.753111e+05
3,hist_mean,5.155611e+05
4,croston_at_origin,2.198482e+05
5,doy_sin,1.532116e+05
6,product_age_days,1.333963e+05
7,hist_std,1.182821e+05
8,dow,1.082408e+05
9,hist_nonzero_rate,1.062095e+05


## Classical models on a stratified sample

ETS and SARIMA are fitted per series, so only on a sample. Their assumptions (a continuous, roughly Gaussian process) fail on intermittent series, which is a result worth reporting rather than a bug to hide.

In [5]:
from src.forecasting.classical import forecast_classical, sample_series
sample = sample_series(cfg, panel)
cl = forecast_classical(cfg, panel, fold, sample)
print('fit failures:', cl.n_failed)
for n in cl.notes: print('note:', n)

fit failures: {'ets': 0, 'sarima': 0}
note: Fitted on a stratified sample of 28 of 600 series.
note: Both models are unconstrained and can forecast negative demand on intermittent series; predictions are clipped at zero.
